# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv('clientes_ecommerce.csv')
# TODO: mostrar 10 primeras filas
print("Filas duplicadas: ", df.duplicated().sum())
# TODO: consultar shape, dtypes, info y número de duplicados

Filas duplicadas:  0


In [3]:
display(df.head(10))
print("Dimensiones:", df.shape)
print(df.dtypes)
df.info()

,cliente_id,edad,antiguedad_meses,ingresos_anuales,gasto_mensual,num_compras,devoluciones,dispositivo,zona,nivel_fidelidad,satisfaccion,compra_proximo_mes
0,20001,67,102.0,9000.00,60.73,7,3,Móvil,Oeste,Oro,5,No
1,20002,28,93.0,35683.86,131.74,6,1,Móvil,Este,Bronce,2,No
2,20003,19,87.0,37152.19,55.33,8,1,Ordenador,Este,Plata,5,Sí
3,20004,55,79.0,40158.29,24.33,7,0,Ordenador,Norte,Bronce,4,Sí
4,20005,39,3.0,27317.06,88.78,5,0,Móvil,Sur,Bronce,4,No
5,20006,45,25.0,32627.01,59.77,9,2,Móvil,Este,Bronce,5,No
6,20007,22,105.0,44101.68,109.67,8,0,Móvil,Norte,Oro,5,Sí
7,20008,39,90.0,44882.10,57.56,7,2,Ordenador,Norte,Plata,2,Sí
8,20009,55,56.0,55299.19,49.94,6,2,Móvil,Sur,Bronce,5,No
9,20010,38,93.0,9000.00,110.69,10,2,Ordenador,Centro,Oro,4,Sí


Dimensiones: (400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona          

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

400 muestras y 12 columnas. El target es compra_proximo_mes. cliente_id identifica a cada cliente. No hay filas duplicadas.

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [4]:
df.isna() #devuelve todo el dataframe entero y pone true donde haya nan


# TODO: ausentes por columna
ausentes =df.isna().sum()
print(ausentes)
# TODO: porcentaje de ausentes por columna

porcentaje = df.isna().mean().mul(100)

print(porcentaje)
# TODO: filas con algún valor ausente

filas_incompletas = df.isna().any(axis=1).sum() #axis 1 se refiere a las filas si fuese axis 0 se refiere a las columnas busc cualquiera en las filas que tiene true y cuentamelo entonces dice cuantas filas que tiene algun campo nan

print("hola bb")
print(filas_incompletas)







cliente_id            0
edad                  0
antiguedad_meses      5
ingresos_anuales      4
gasto_mensual         3
num_compras           0
devoluciones          0
dispositivo           0
zona                  0
nivel_fidelidad       0
satisfaccion          0
compra_proximo_mes    0
dtype: int64
cliente_id            0.00
edad                  0.00
antiguedad_meses      1.25
ingresos_anuales      1.00
gasto_mensual         0.75
num_compras           0.00
devoluciones          0.00
dispositivo           0.00
zona                  0.00
nivel_fidelidad       0.00
satisfaccion          0.00
compra_proximo_mes    0.00
dtype: float64
hola bb
12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1,25 % |
| `ingresos_anuales` | 4 | 1,00 % |
| `gasto_mensual` | 3 | 0,75 % |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No, se perderian 12 filas (un 3 %) que solo tienen un dato vacio. Es mejor imputar.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [5]:
# TODO: mínimo y máximo de edad
print("Edad Minima: ", df['edad'].min())
print("Edad maxima: ", df['edad'].max())

# TODO: localizar registros con edad > 120 loc es con corchetes[]
#.        filas.                    columnas saco cliente id y edad de todos los mayores de 120 años
df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad Minima:  18
Edad maxima:  195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta: ... ya veremos como lo arreglamos

Lo pondria como NaN y luego lo imputaria con la mediana. No se cambia por 95 porque no sabemos el valor real, podria ser 19, 59 o cualquier otro.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | numeros |
| Categóricas nominales | dispositivo zona   |
| Categóricas ordinales |porque tiene orden jerarquia yo se que el bronce es el nivel de |
| Identificador | id que no aporta nada |
| Target | si compra o no es lo que se quiere predecir |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta: zona no tiene orden, ninguna zona es mayor que otra. nivel_fidelidad si tiene orden (Bronce, Plata, Oro, Platino). cliente_id no se incluye porque es solo un numero para identificar al cliente y no ayuda a predecir.

| Tipo | Columnas |
|---|---|
| Numéricas | edad, antiguedad_meses, ingresos_anuales, gasto_mensual, num_compras, devoluciones, satisfaccion |
| Categóricas nominales | dispositivo, zona |
| Categóricas ordinales | nivel_fidelidad |
| Identificador | cliente_id |
| Target | compra_proximo_mes |

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [6]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])
# TODO: y = ... (mapear No:0, Sí:1)
y = df['compra_proximo_mes'].map({"No":0,"Sí":1})

# TODO: X.shape, y.shape, y.value_counts()
print(X.shape)
print(y.shape)
print(y.value_counts())

(400, 10)
(400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: 10

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [7]:
# TODO: completar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify= y #segun la semilla que le pongamos empieza por un numero u otro y de yes el objetivo es la segunda y va a conservar el porcentaje de la segunda
 )
# TODO: mostrar las dimensiones de los cuatro conjuntos

print("Entrenamiento: ", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


#imputacion sustituir campos NaN puedo sustituir el valor NaN por el valor medio de los satos suponemos que tengo una columna con la edad de los clientes sustituiriamos el campo nAn por la edad media del cliente porque no se puede entrenar a un modelo con un campo null
#tambien se puede usar la mediana se coge la edad mediana por ejemplo con el señor de los 200 años cuando hay outliers tiene mas sentido utilizar la mediana en lugar de la media
#usar la moda el valor que mas se repite
#un valor constante cada vez que te encuentres un nan reemplaalo con 40 se define 40 como un dato fijo que yo elijo 


#codificacion transformar datos de una forma a otra forma 
#one hot encoding codificar valores categoricos de forma numerica: tenemos caracteristica que es ciudad tenemos madrid barcelona y valencia 

# ciudad: madrid, barcelona , valencia
#codificacion ordinal  oro plata bronce

#escalado tipos:     devuelve un valor entre 0 y 1    1                                [20,30,40,50,60]
#normalizacion min max valores numericos rango [0-1] x= x- xmin                        xmin=20
                                                    # -----------  =0.25               xmax=60
                                                    # xmax- xmin                        x=30         es importante guardar la proporcion
#z-score media: 0
#desviacion tipica: 1

#z= x - m.    x valor originalk.  z= 0,71 
#----------    m media ≠ 40
#.   o.        o desviacion  la raiz  de 20 primera edad - 40 la media al cuadrado +  y asi todas las edades al cuadrado entre 5 14,14

Entrenamiento:  (320, 10) (320,)
Test:  (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: 320 en entrenamiento y 80 en prueba. stratify=y hace que los dos conjuntos tengan la misma proporcion de Si y No. Dividimos antes para que la prueba no influya en las transformaciones (data leakage).

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [8]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
print(X_train.loc[df['edad']>100, ["edad"]])
X_train.loc[df['edad']>100, ["edad"]]=np.nan
print(X_train['edad'].isna().sum())

print(X_test.loc[df['edad']>100, ["edad"]])
X_test.loc[df['edad']>100, ["edad"]]=np.nan
print(X_test['edad'].isna().sum())
# TODO: contar ausentes en edad en entrenamiento y prueba

     edad
137   195
1
Empty DataFrame
Columns: [edad]
Index: []
0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [9]:
# TODO: imputer_numerico = SimpleImputer(...) si imputa
"""
strategy: 
Media mean
mediana median
moda most_frequent
valor constante: constant ---> fill_value= x
"""
#imputer_numerico = SimpleImputer(strategy="constant", fill_value=20) hay que especificar el valor por el que sequiere imputar

imputer_numerico = SimpleImputer(strategy="median")
# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | No le afectan los valores extremos como el 195 |
| antiguedad_meses | Mediana | Es numerica y la mediana es mas robusta |
| ingresos_anuales | Mediana | Hay valores altos que moverian la media |
| gasto_mensual | Mediana | Tiene cola a la derecha |
| dispositivo | Moda | Es categorica, se usa la mas frecuente |
| zona | Moda | Es categorica, se usa la mas frecuente |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque el imputador aprenderia de los datos de prueba y eso es data leakage. Se ajusta solo con entrenamiento.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [10]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)#si encuentra una categoria desconocida que la ignora que es lo que hace cuando la ignora volviendo a lo de las ciudades si tiene las 3 ciudades si aparece 1 persona que es de madrid va a a aparecer suponiendo que el orden es m b v pues la de madrid 1 0 0 y si no hay en ninguna 0 0 0 eso es lo que hace handle_unknown
# TODO: aplicar fit_transform a ejemplo
matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)
# ejecuta la funcion fit() localizar las categorias de forma unica entonces va a analizar todas las que hay y se queda coin movil y ordwedaro y tablet solo
#transform() por cada dato hace la transformacion utilizando one hot encoder
print(matriz_ejemplo)
# TODO: consultar get_feature_names_out()
print(encoder_nominal.get_feature_names_out()) #genera el nombre de la columna a partir de la informacion que he utilizado con los datos orginales le estamos pasando un dataframe que se llama dispiotivi y tiene unas categoriascon los campos
#devuelve el nombre de la columna con el nombre de la categoria de forma unica sin repeticiones genera el nombre compuesto de la categorica aue estoy utilizando
#devuelve una lista con 3 columnas se ouede crrear un fata drame nuevo que combine las 2 cosdas devuelve clientes y su dispositivo
pd.DataFrame(matriz_ejemplo, columns=encoder_nominal.get_feature_names_out())

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: 3 columnas, una por cada dispositivo, con 1 si es ese dispositivo y 0 si no. No se pone 0, 1, 2 porque el modelo pensaria que hay un orden que no existe.

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [11]:
#codificacion ordinal asignando una jerarquia bronce es la mas baja y platina es la mas alta no es necesario tenerlo ordenado bronce sera 0 plata sera 1 oro dsera 2 y platino sera 3
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"] # la ejecucion la hace segun este orden pero pone el numero indicado
    #arriba se utiliza one hot encoder porque no hay una logica o jerarquia por ejemplo entre ciudades dispositivos en cambio con el nivekl de fidelidad si
})
#con fit_transform se ejecutan las 2 funciones al mismon tiempo

# TODO: 
encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]])
# TODO: transformar ejemplo_fidelidad y mostrar el resultado
resultado= encoder_ordinal.fit_transform(ejemplo_fidelidad)
print(resultado.ravel()) 

pd.DataFrame({"nivel":ejemplo_fidelidad['nivel_fidelidad'],"codigo":resultado.ravel()})

[2. 0. 3. 1.]


,nivel,codigo
0,Oro,2.0
1,Bronce,0.0
2,Platino,3.0
3,Plata,1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: La posicion de cada nivel: Bronce 0, Plata 1, Oro 2 y Platino 3. No, solo indica el orden, no que la diferencia entre niveles sea la misma.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [12]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]}) #escalar para que todo este a la misma proporcion 2 formas de escalar min-max y z-score
#montar pipelines y entrenar modelo
print(datos)
# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
zscore = StandardScaler()
# TODO: transformar datos con cada uno
print(minmax.fit_transform(datos))
print(zscore.fit_transform(datos))

#esto de abajo no habia que hacerlo
comparacion = {"datos": datos['valor'], "minmax" : minmax.fit_transform(datos).ravel(),
               "zscore" : zscore.fit_transform(datos).ravel()}

#esto si
print(pd.DataFrame(comparacion))
# TODO: mostrar tabla comparativa

   valor
0     10
1     20
2     30
3     40
4     50
[[0.  ]
 [0.25]
 [0.5 ]
 [0.75]
 [1.  ]]
[[-1.41421356]
 [-0.70710678]
 [ 0.        ]
 [ 0.70710678]
 [ 1.41421356]]
   datos  minmax    zscore
0     10    0.00 -1.414214
1     20    0.25 -0.707107
2     30    0.50  0.000000
3     40    0.75  0.707107
4     50    1.00  1.414214


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | -1.414 |
| 20 | 0.250 | -0.707 |
| 30 | 0.500 | 0.000 |
| 40 | 0.750 | 0.707 |
| 50 | 1.000 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: Min-Max da entre 0 y 1. Despues de estandarizar la media es 0. Si, con StandardScaler salen negativos los valores por debajo de la media. Para distancias usaria StandardScaler porque le afectan menos los valores extremos.

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [13]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

#que no esta aqui lo que no esta en x compra proximo mes

# TODO: comprobar que las columnas existen en X_train

In [14]:
# comprobar que las columnas existen en X_train
for col in numericas + nominales + ordinales:
    print(col, col in X_train.columns)

edad True
antiguedad_meses True
ingresos_anuales True
gasto_mensual True
num_compras True
devoluciones True
satisfaccion True
dispositivo True
zona True
nivel_fidelidad True


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: ingresos_anuales (miles de euros) frente a devoluciones o satisfaccion (de 0 a 5). No es obligatorio, los arboles de decision no lo necesitan. satisfaccion es ordinal de 1 a 5 aunque la tratemos como numerica.

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [15]:
# TODO: #transformaciones necesaroas para pasos numericos
pipeline_numerico = Pipeline(steps=[
    ("imputer", imputer_numerico), #nombre da igual
    ("scaler", zscore)
])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: Porque el escalador no funciona con valores NaN, primero hay que rellenarlos

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [16]:
# TODO: 
pipeline_nominal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", encoder_nominal)
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [17]:
pipeline_ordinal = Pipeline(steps=[ #sonia ha decidido modificar el ejerccio 10 pero yo lo he dejado aqui
    ("imputer", imputer_categorico),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
        ))
        ])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [18]:
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
    ])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [19]:
#columns= ['edad', 'antiguedad_meses', 'ingresos_anuales', 'gasto_mensual', 'num_compras', 'devoluciones', 'dispositivo', 'zona, 'nivel_fidelidad']
X_train_preparado = preprocesador.fit_transform(X_train)
print(X_train_preparado[0]) # fit procesador saca las equivalencias solo se usa para entrenar al modelo
# TODO: las predicciones se hacen sobre tren comprueba el mapeo que ha hecho antes y lo aplica a los datos nuevos
X_test_preparado = preprocesador.transform(X_test)
print(X_test_preparado[0])

[-0.06872324 -1.09556504  0.65104519 -1.08687713  0.27794508 -0.18520516
 -0.25429301  0.          1.          0.          0.          0.
  1.          0.          0.          3.        ]
[ 0.05311298 -1.54065282 -0.99459538 -0.88736384  1.01300645  1.58391879
 -0.25429301  1.          0.          0.          0.          1.
  0.          0.          0.          1.        ]


**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: Porque volveria a aprender de los datos de prueba (data leakage). fit() aprende medianas, modas, valores de escalado y categorias. Si aparece una categoria nueva se pone todo a 0 gracias a handle_unknown="ignore".

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [20]:
#NO HACE FALTA DICHO EL 05 NOV a las 18:57

# TODO: mostrar X_train_preparado.shape y X_test_preparado.shape
# TODO: contar np.isnan(...) en ambas matrices
# TODO: consultar dtype y get_feature_names_out()
# TODO: verificar filas de matrices frente a len(y_train), len(y_test)
# TODO: verificar que entrenamiento y prueba tienen igual número de columnas

**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

Respuesta: ...

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [21]:
# TODO: columnas_finales = preprocesador.get_feature_names_out()
columnas_finales = preprocesador.get_feature_names_out()
# TODO: df_train_preparado = pd.DataFrame(..., columns=..., index=X_train.index)
df_train_preparado = pd.DataFrame(X_train_preparado, columns=columnas_finales, index=X_train.index)
# TODO: df_test_preparado = pd.DataFrame(..., columns=..., index=X_test.index)
df_test_preparado = pd.DataFrame(X_test_preparado, columns=columnas_finales, index=X_test.index)
# TODO: mostrar head() de ambos
display(df_train_preparado.head())
display(df_test_preparado.head())

,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
360,-0.068723,-1.095565,0.651045,-1.086877,0.277945,-0.185205,-0.254293,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
356,-0.677904,1.129874,1.422884,-1.092939,0.277945,-1.069767,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
138,0.845048,-1.540653,0.907575,0.537957,-0.457116,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,-1.408922,0.447406,0.476223,0.303390,1.380537,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
216,-0.495150,1.456272,-0.493983,1.466735,0.645476,-1.069767,1.495680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.053113,-1.540653,-0.994595,-0.887364,1.013006,1.583919,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,-0.495150,1.040856,-0.185324,-0.504414,-0.089586,-1.069767,-0.254293,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,1.697902,-1.036220,0.611217,-0.253507,-0.457116,1.583919,-0.254293,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.784130,-1.095565,0.373354,-0.176812,-1.559708,1.583919,0.620694,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,-0.312396,0.269371,-1.092628,1.067445,1.013006,-1.069767,1.495680,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
